# Starbucks Capstone Challenge

## Introduction

- This data set contains simulated data that mimics customer behavior on the Starbucks rewards mobile app. Once every few days, Starbucks sends out an offer to users of the mobile app. An offer can be merely an advertisement for a drink or an actual offer such as a discount or BOGO (buy one get one free). Some users might not receive any offer during certain weeks.

- Not all users receive the same offer, and that is the challenge to solve with this data set.

- Your task is to combine transaction, demographic and offer data to determine which demographic groups respond best to which offer type. This data set is a simplified version of the real Starbucks app because the underlying simulator only has one product whereas Starbucks actually sells dozens of products.

- Every offer has a validity period before the offer expires. As an example, a BOGO offer might be valid for only $5$ days. You'll see in the data set that informational offers have a validity period even though these ads are merely providing information about a product; for example, if an informational offer has 7 days of validity, you can assume the customer is feeling the influence of the offer for 7 days after receiving the advertisement.

- You'll be given transactional data showing user purchases made on the app including the timestamp of purchase and the amount of money spent on a purchase. This transactional data also has a record for each offer that a user receives as well as a record for when a user actually views the offer. There are also records for when a user completes an offer.

- Keep in mind as well that someone using the app might make a purchase through the app without having received an offer or seen an offer.


## Data Sets

The data is contained in three files:

* portfolio.json - containing offer ids and meta data about each offer (duration, type, etc.)
* profile.json - demographic data for each customer
* transcript.json - records for transactions, offers received, offers viewed, and offers completed

Here is the schema and explanation of each variable in the files:

### profile.json

* `age` (int) - age of the customer 
* `became_member_on` (int) - date when customer created an app account
* `gender` (str) - gender of the customer (note some entries contain '`O`' for other rather than M or F)
* `id` (str) - customer id
* `income` (float) - customer's income

### portfolio.json

* `id` (string) - offer id
* `offer_type` (string) - type of offer ie BOGO, discount, informational
* `difficulty` (int) - minimum required spend to complete an offer
* `reward` (int) - reward given for completing an offer
* `duration` (int) - time for offer to be open, in days
* `channels` (list of strings)



### transcript.json

* `event` (`str`) - record description (ie transaction, offer received, offer viewed, etc.)
* `person` (`str`) - customer id
* `time` (`int`) - time in hours since start of test. The data begins at time `t=0`
* `value` - (`dict of strings`) - either an offer id or transaction amount depending on the record

# Load Lib

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
#!pip install tabulate

In [ ]:
from importlib import reload
from tabulate import tabulate

In [ ]:
import pandas as pd
import numpy as np
import math
import json

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
sns.color_palette()

In [ ]:
plt.style.use('seaborn')
plt.rcParams['figure.figsize'] = (10,5)
plt.rcParams['lines.linewidth'] = 2


font = {'family': 'serif',
        'color':  '#2E86C1',
        'weight': 'normal',
        'size': 20,
        }

base_color = sns.color_palette()[0]

In [ ]:
#!pip install print
#!pip install confusion_matrix

In [ ]:
# Confusion-matrix rendering uses scikit-learn; no external helper package is required.

In [ ]:
reload(p)

# Import Datasets

In [ ]:
# read in the json files
portfolio = pd.read_csv('data/portfolio.csv')
profile = pd.read_csv('data/profile.csv')
transcript = pd.read_csv('data/transcript.csv')

In [ ]:
profile.head(2)

In [ ]:
portfolio.head(2)

In [ ]:
transcript.head(2)

# Assessing

## profile

In [ ]:
# check the type on columns
print('check the type on columns')

profile.info()

In [ ]:
# check the null values
print('check the null values')

profile.isnull().sum()

In [ ]:
print('check the age column')

# check the age column
print(profile.age.value_counts(), '\n')

print('describe of age column')

# check the describe of age column
print(profile.age.describe(), '\n')

print('check the age 118')

# check the age 118
x = profile[profile.age == 118]
print(x[:2])
x.age.value_counts()

In [ ]:
# check income column
print('describe of income column')
profile.income.describe()

In [ ]:
# check duplicate
print('check duplicate')
profile[profile.duplicated()]

## portfolio

In [ ]:
portfolio.head(2)

In [ ]:
# check columns type
print('check columns type')

portfolio.info()

In [ ]:
# check offer_type column

print('check offer_type column')
portfolio.offer_type.value_counts()

## transcript

In [ ]:
transcript.head(2)

In [ ]:
# check column type
print('check column type')
transcript.info()

In [ ]:
# check event column
print('check event column')
transcript.event.value_counts()

In [ ]:
# check time column
print('check time column')
transcript.time.value_counts()

In [ ]:
# check value column
print('check value column')
transcript.value

## Observation

**Profile Dataset**
- [x] `became_member_on` not date
- [x] `gender`there are $2175$ Missing value
- [x] `income`there are $2175$ Missing value
- [x] `age` there are $2175$ persons have 118 years it seem to be outlier. we can also notice that is located in same row with `gender` and `income` that have `NaN` value
- [x] `id` change his name to `customer_id`
- [x] `gender` need to encode
- [x] `become_member_on` need to encode

---

**Portfolio Dataset**
- [x] `id` change his name to `offer_id`
- [x] `channels` need to encode
- [x] `offer_type` need to encode
- [x] `offer_id` create column has offer name like `offer_1`

---

**Transcript Dataset**
- [x] `time` convert to days
- [x] `person` change his name to `customer_id`
- [x] `value` extract `offer_id` and `amount`
- [x] `event` need to encode

# Data Wrangling & Feature Enginnering

## profile

### 
**Define**: `became_member_on` not date

In [ ]:
# check became_member_on column
print('check became_member_on column')

print(profile.became_member_on.value_counts())

# Code: convert date string to date
print('Code:convert date string to date')

profile['became_member_on'] =\
    pd.to_datetime(profile.became_member_on, format="%Y%m%d")
print('done')


# test : convert date string to date
print('test:convert date string to date')
print(profile.became_member_on)

### 
**Define**: `age` there are $2175$ persons have 118 years it seem to be outlier. we can also notice that is located in same row with `gender` and `income` that have `NaN` value

**Define**: `gender`there are $2175$ Missing value

**Define**: `income`there are $2175$ Missing value

In [ ]:
# code: drop nan value rows
print('code: drop nan value rows')
profile.dropna(inplace=True)
profile.reset_index(drop=True, inplace=True)
print('done')

# test: 
print('test:')
profile.isnull().sum()

### 
**Define**: id change his name to customer_id

In [ ]:
# check columns name
print('check columns name')
print(profile.columns)


# code: change id name
try:
    print('code: change id name')
    profile.rename(columns={"id": "customer_id"}, inplace=True)
except Exception as e: print(e)
else:
    print('done')


# test:
print('test: columns name')
print(profile.columns)

### 
**Define**: `gender` need to encode

In [ ]:
profile.gender.value_counts()

In [ ]:
encoder = OneHotEncoder(sparse=False)


try:
    # code: encode gender column
    print('code: encode gender column')
    onehot = encoder.fit_transform(profile[['gender']])
    # print(onehot)

    profile[['Female', 'Male', 'other']] = onehot
    
except Exception as e: print(e)
else: print('done')

In [ ]:
profile.head(2)

### 
**Define**`become_member_on` need to encode

In [ ]:
# code: encode become_member_on
print('code: encode become_member_on')
try:
    profile['become_member_on_year'] = profile.became_member_on.dt.year.astype(str)
    profile['become_member_on_month'] = profile.became_member_on.dt.month.astype(str)
    profile['become_member_on_day'] = profile.became_member_on.dt.day.astype(str)
except Exception as e: print(e)
else: print('done')

In [ ]:
profile

In [ ]:
# drop become_member_on column
print('drop become_member_on column')

profile.drop('became_member_on', axis=1, inplace=True)

## portfolio 

### 
**Define**: `id` change his name to `offer_id`

In [ ]:
# check columns name
print('check columns name')
print(portfolio.columns)


# code: change id name
try:
    print('code: change id name')
    portfolio.rename(columns={"id": "offer_id"}, inplace=True)
except Exception as e: print(e)
else:
    print('done')


# test:
print('test: columns name')
print(portfolio.columns)

### 
**Define**: `channels` need to encode

In [ ]:
# check the type
print('check the type')
type(portfolio.channels[0])

In [ ]:
# return all channels
print('return all channels')
try:
    print(portfolio.channels.value_counts())
except: pass

In [ ]:
portfolio.head(2)

In [ ]:
# code: split channels column
print('code: split channels column')

def split_channels(channels_list):
    """split channels to columns"""
    if channel in channels_list:
        return 1
    else:
        return 0
try:
    all_channels = ['web', 'email', 'mobile', 'social']
    for channel in all_channels:
        portfolio.loc[:, channel] = portfolio.channels.apply(split_channels)
    
    
except: pass

else: print('done')


# test
print('test: split channels column')

portfolio.head(2)

In [ ]:
try:
    print('Drop channels column')
    portfolio.drop('channels', axis=1, inplace=True)
except: pass
else: print()

### 
**Define**: `offer_type` need to encode


In [ ]:
# return all offer_type

print('return all offer_type')
offer_type = portfolio.offer_type.unique()
offer_type

In [ ]:
portfolio.head(2)

In [ ]:
# code: split offer_type column
print('code: split offer_type column')

def split_offer_type(val):
    """split offer_type column"""
    if val == offer:
        return 1
    else:
        return 0

try:
    for offer in offer_type:
        portfolio[offer] = portfolio.offer_type.apply(split_offer_type)
    
except: pass
else: print('done')
    

# test
print('test: split offer_type column')
portfolio.head(2)

### 
**Define** `offer_id` create column has offer name like offer_1

In [ ]:
portfolio

In [ ]:
# code: create offer_name column
print('code: create offer_name column')

try: 
    lst = range(1,portfolio.shape[0]+1)
    x = [f"offer_{x}" for x in lst]
    portfolio['offer_name'] = x

except: pass

else: 
    print('done')
    print(portfolio[['offer_id','offer_name']])


## transcript

### 
**Define**: `time` convert to days

In [ ]:
transcript.head(5)

In [ ]:
# check time column
print('check time column')
print(transcript.time.value_counts().sort_index()[:5])

# check the type
print('check the type')
print(type(transcript.time[0]))

In [ ]:
def convert_time_day(val):
    """convert from time to days"""
    return val / 24

try:
    # code: convert time to days
    print('code: convert time to days')

    transcript['days'] = transcript.time.apply(convert_time_day)

except Exception as e: print(e)

else: print('done')

In [ ]:
# test: convert time to days
print('test: convert time to days')

transcript.days[-10:]

### 
**Define**: `person` change his name to `customer_id`

In [ ]:
# return columns name

print('return columns name')
print(transcript.columns)

# code: change column name
try:
    print('code: change column name')
    transcript.rename(columns={"person": "customer_id"}, inplace=True)
except: pass
else: print('done')
    
# test: change column name
print('return columns name')
print(transcript.columns)

### 
**Define**: `value` extract `offer_id` and `amount`

In [ ]:
# check value column
print('check value column')
transcript.value

In [ ]:

import ast
def string_to_dict(val):
    return ast.literal_eval(val)

transcript.value = transcript.value.apply(string_to_dict)

In [ ]:
# return the keys in value column
print('return the keys in value column')

keys = []
def get_keys(val):
    """return available keys"""
    if list(val.keys()) not in keys:
        keys.append(list(val.keys()))
    
    
transcript.value.apply(get_keys)

keys = [ item for elem in keys for item in elem]
print(keys)

In [ ]:
transcript.tail(10)

In [ ]:
def split_key(val):
    """split key column"""
    if list(val.keys())[0].startswith(key[:2]):
        return list(val.values())[0]
    else:
        return 0
    
try:  
    # code: split value column
    print('code: splite value column')
    
    for key in keys[1:3]:
        transcript[key] = transcript.value.apply(split_key)
except: pass

else: print('done')
    
# test: split value column
print('test: splite value column')
transcript.tail(10)



In [ ]:
# drop value column
print('drop value column')

transcript.drop('value', axis=1, inplace=True)

# test
print('test')

transcript.columns

### 
**Define**: `event` need to encode

In [ ]:
# return unique value in event column
print('return unique value in event column')

event_list = transcript.event.unique()
event_list

In [ ]:
transcript.tail(50)

In [ ]:
def split_event(val):
    """split event column"""
    if val == event:
        return 1
    else:
        return 0

try:
    # code: split event column
    print('code: split event column')
    
    for event in event_list:
        transcript[event] = transcript.event.apply(split_event)
except: pass
else: print('done')
    

# test: split event column
print('test: split event column')

transcript.tail(50)

In [ ]:
# rename columns
print('rename columns')

transcript.rename(columns={ "offer received": "offer_received", 
                            "offer viewed": "offer_viewed",
                            "offer completed": "offer_completed"},
                 inplace=True)

# test: rename columns
print('test: rename columns')
transcript.columns

## Create offer dataframe

create offer dataframe from people that have offers

In [ ]:
# create offer dataframe 
offer = transcript[transcript.event != 'transaction']
offer.head()

## Create successful offer column

**Note**: that the offer be success if a customer completed an offer and viewed it and if not viewed the offer become not success offer

In [ ]:
# check if the offer is success or not

# Group-by value counts in event column 
offer_customer = offer.pivot_table(index=['customer_id','offer_id'],\
                 columns='event', aggfunc='size', fill_value=0)
offer_customer.reset_index(level=[0,1], inplace=True)

# creat successful_offer column 
offer_customer['successful_offer'] =\
    offer_customer['offer completed'] * offer_customer['offer viewed']

offer_customer['successful_offer'] = offer_customer['successful_offer'].apply(lambda x: 1 if x > 0 else 0)

## Merge datasets
create overall dataset that a combination from offer_customer dataset and profile dataset and portoflio dataset

In [ ]:
# merge dataset profile and portolio and offer_customer
overall_df = pd.merge(offer_customer,portfolio,on='offer_id', how='left')
overall_df = pd.merge(overall_df,profile,on='customer_id')
overall_df

# Export clean data

In [ ]:
# export clean data
print('export clean data')

profile.to_csv('clean_profile.csv', index=False)
portfolio.to_csv('clean_portfolio.csv', index=False)
transcript.to_csv('clean_transcript.csv', index=False)
overall_df.to_csv('overall_df.csv', index=False)

print('done')

# Exploratory analysis

In [ ]:
portfolio = pd.read_csv('clean_portfolio.csv')
profile = pd.read_csv('clean_profile.csv')
transcript = pd.read_csv('clean_transcript.csv')
overall_df = pd.read_csv('overall_df.csv')

## Profile

In [ ]:
profile['become_member_on_month'] = profile.become_member_on_month.astype(str)
profile['become_member_on_year'] = profile.become_member_on_year.astype(str)
profile['become_member_on_day'] = profile.become_member_on_day.astype(str)

### Gender Distribution

In [ ]:
gender_count = [profile.Female.sum(), profile.Male.sum(), profile.other.sum()]
x_label = ['Female', 'Male', 'Other']
sns.barplot(y = gender_count, x = x_label , color=base_color);
plt.title('Gender Distribution', fontdict=font);


> we can notice that most of users are Male

### Income Distribution

In [ ]:
bin_edges = np.arange(0, profile['income'].max()+1, 10000)

sns.histplot(profile['income'],bins=10);
plt.xticks(bin_edges)
plt.xlim(profile.income.min()-5000, profile.income.max()+5000)
plt.title('Income Distribution', fontdict=font);

> We can notice that the most income between $50000 \to 70000$

### Age Distribution

In [ ]:
bin_edges = np.arange(0, profile['age'].max()+10, 10)
sns.histplot(profile['age'],bins=10);
plt.xticks(bin_edges)
plt.xlim(profile.age.min()-5, profile.age.max()+10)
plt.title('Age Distribution', fontdict=font);

> We can notice that the most age is between 40 to 75

### New Member based of Year

In [ ]:
cat_order = profile['become_member_on_year'].value_counts().index
sns.countplot(data = profile, x = 'become_member_on_year', color=base_color, 
              order=cat_order)
plt.title('New Member based of Year', fontdict=font)

# add annotations
n_points = profile.shape[0]
cat_counts = profile['become_member_on_year'].value_counts()
locs, labels = plt.xticks() # get the current tick locations and labels

# loop through each pair of locations and labels
for loc, label in zip(locs, labels):

    # get the text property for the label to get the correct count
    count = cat_counts[label.get_text()]
    pct_string = '{:0.1f}%'.format(100*count/n_points)

    # print the annotation just below the top of the bar
    plt.text(loc, count-8, pct_string, ha = 'center', color = 'k')


> 2017 has the highest number of new members.

### New Member based of Month

In [ ]:
cat_order = profile['become_member_on_month'].value_counts().index
sns.countplot(data = profile, x = 'become_member_on_month', color=base_color, 
              order=cat_order)
plt.title('New Member based of Month', fontdict=font)

# add annotations
n_points = profile.shape[0]
cat_counts = profile['become_member_on_month'].value_counts()
locs, labels = plt.xticks() # get the current tick locations and labels

# loop through each pair of locations and labels
for loc, label in zip(locs, labels):

    # get the text property for the label to get the correct count
    count = cat_counts[label.get_text()]
    pct_string = '{:0.1f}%'.format(100*count/n_points)

    # print the annotation just below the top of the bar
    plt.text(loc, count-8, pct_string, ha = 'center', color = 'k')


> We can notice grow in member based on month is almost equal

### New Memeber based on Year and Gender

In [ ]:
sns.countplot(data = profile, x = 'become_member_on_year', hue = 'gender')
plt.title('New Member based of Year and Gender', fontdict=font);

> We can notice that most new member are Male

### Income based on Gender

In [ ]:
sns.violinplot(data = profile, y = 'income', x = 'gender');
plt.title('Income based on Gender', fontdict=font);

In [ ]:
sns.histplot(data=profile, x="income", hue="gender", multiple="stack", 
            bins=10);

bin_edges = np.arange(0, profile['income'].max()+1, 10000)

plt.xticks(bin_edges)
plt.xlim(profile.income.min()-5000, profile.income.max()+5000)
plt.title('Income Distribution Based on Gender', fontdict=font);

> We can notice that the count Female gender is the highest one

## transcript

### Event Distribution

In [ ]:
# create event list
x_value = ['Offer received','Offer viewed','Transaction','Offer completed']
y_value = [transcript.offer_received.sum(), transcript.offer_viewed.sum(),
    transcript.transaction.sum(), transcript.offer_completed.sum()]

# sort values
x_value = [x for _,x in sorted(zip(y_value,x_value), reverse=True)]
y_value.sort(reverse=True)

# show graph
sns.barplot(y = y_value, x = x_value , color=base_color);
plt.title('Event Distribution', fontdict=font);

> We can notice that most of offers are just received and viewed

### Offers Distribution

In [ ]:
table = [
    ['Number of customer made transaction' ,
         transcript[transcript.transaction == 1].customer_id.count(),
    '100%'],
    
    
    ['Number of unique customers made transaction',
        transcript[transcript.transaction == 1].customer_id.nunique(),
    f"{round((16578/138953)*100, 2)}%"],
    
    
    ['Number of customers who recived offers', 
        transcript[transcript.transaction != 1].customer_id.count(),
    '100%'],
    
    
    ['Number of unique customers who recived offers',
        transcript[transcript.transaction != 1].customer_id.nunique(),
    f"{round((16994/167581)*100, 2)}%"]
        
        ]

header = ['Notice', 'Number', 'Percentage']

print(tabulate(table,header, tablefmt="simple"))

> We can notice that there are alot of repeated offers to the same customer
also we can notice that the customers who made transaction mostly are the same customers no new customer come

## Overall dataframe

### Distribution of Successfule offers

In [ ]:
x = overall_df['successful_offer'].value_counts()
x.index = ['Failure', 'Success']

ax = sns.countplot(data = offer_customer, x = 'successful_offer', color = base_color)
plt.title('Distribution of Successfule offers', fontdict=font);
ax.set_xticklabels(['Failure', 'Success']);
ax.set_xlabel('Successfule Offers');

# add annotations
n_points = overall_df.shape[0]
cat_counts = x
locs, labels = plt.xticks() # get the current tick locations and labels

# loop through each pair of locations and labels
for loc, label in zip(locs, labels):

    # get the text property for the label to get the correct count
    count = cat_counts[label.get_text()]
    pct_string = '{:0.1f}%'.format(100*count/n_points)

    # print the annotation just below the top of the bar
    plt.text(loc, count-8, pct_string, ha = 'center', color = 'k')

> The Percentage of failure offers greater than success offers

### Distribution of Success based on offer type

In [ ]:
plt.subplots(figsize=(15,5))
ax = sns.countplot(data = overall_df, x = 'offer_name', hue = 'successful_offer', 
                  palette=['#AAB7B8', '#5DADE2'])
ax.legend(['Failure', 'Success'], title = 'Offer', );
plt.title('Distribution of Success based on offer type', fontdict=font);
plt.xticks(rotation=20);
plt.xlabel('Offer Name');

> There are alot of offers need to review because is not success like offer_8. <br>
offer_7, offer_9, offer_6 have a successeful story

### Distribution of Offers based on Gender

In [ ]:
plt.subplots(figsize=(15,5))
ax = sns.countplot(data = overall_df, x = 'offer_name', hue = 'gender', 
                   )
plt.title('Distribution of Offers based on Gender', fontdict=font);
plt.xticks(rotation=20);
plt.xlabel('Offers Name');

> We can notice that male have more offers then female and that is normal because the count of male is more than female

### Distribution of Success based on Gender

In [ ]:
ax = sns.countplot(data = overall_df, x = 'gender', hue = 'successful_offer', 
                   palette=['#AAB7B8', '#5DADE2'])
ax.legend(['Failure', 'Success'], title = 'Offer', );
plt.title('Distribution of Success based on gender', fontdict=font);
plt.xlabel('Gender');

> The offers is failed with male more than female

### Distribution of of Success offers based on Gender

In [ ]:
succes_offer = overall_df[overall_df.successful_offer == 1]
plt.subplots(figsize=(15,5))
ax = sns.countplot(data = succes_offer, x = 'offer_name', hue = 'gender', 
                   )
plt.title('Distribution of of Success offers based on Gender', fontdict=font);
plt.xticks(rotation=20);
plt.xlabel('Offers Name');

> We can notice that the success offer in female is more than the other despite of the male in data set is more than female

### Income Distribution Based on Success

In [ ]:
ax = sns.histplot(data=overall_df, x="income", hue="successful_offer", multiple="stack", 
            bins=10);

bin_edges = np.arange(0, profile['income'].max()+1, 10000)

plt.xticks(bin_edges)
plt.xlim(profile.income.min()-5000, profile.income.max()+5000)
plt.title('Income Distribution Based on Success', fontdict=font);

> We can notice that the income does not have any role in the reject offers

# Modeling

I will build a model that **predicts whether or not someone will respond to an offer**. 

In [ ]:
overall_df = pd.read_csv('overall_df.csv')

## Data Preparation

In [ ]:
overall_df.head()

In [ ]:
# create x and y data
X = overall_df.drop(['customer_id', 'offer_id', 
                     'offer_type', 'gender', 'offer_name', 'successful_offer', 
                    'offer received', 'social', 'discount', 'other', # remove C-1 binary variables
                     'become_member_on_year', 'become_member_on_month',
                     'become_member_on_day'], axis=1)
y = overall_df.loc[:, 'successful_offer']

## ML Pipline

In [ ]:
X.iloc[:5]

In [ ]:
X.shape

### Train Test Split

In [ ]:
X_train,X_test,y_train,y_test =\
    train_test_split(X, y, 
                     test_size=0.3, random_state=42)


### Create Transformers

In [ ]:
all_column = X.columns

# feature scaling
trf1 = ColumnTransformer([
    ('scale', MinMaxScaler(), all_column)
])

trf2 = LogisticRegression(solver='liblinear',random_state=42)

### Create Pipeline

In [ ]:
pipe = Pipeline([
    ('trf1', trf1),
    ('trf2', trf2),
])

#fit data
pipe.fit(X_train, y_train);

In [ ]:
# check cross validation 
scoring = ['accuracy', 'precision', 'recall']

cross_val = cross_validate(pipe, X, y, cv=10, scoring=scoring)

### Check the Accurecy

In [ ]:
# cross validation accurecy
table = []
header = ["Matrices", 'Score']
table.append(['Accuracy', cross_val['test_accuracy']])
table.append(['Precision', cross_val['test_precision']])
table.append(['Recall', cross_val['test_recall']])

print(tabulate(table, header, tablefmt="fancy_grid"))

---

In [ ]:
# pipeline accurecy
pipe.score(X_test, y_test)

In [ ]:
y_pred = pipe.predict(X_test)
cf_matrix = confusion_matrix(y_test, y_pred)

group_names =  ['True Neg','False Pos','False Neg','True Pos']
categories = ['Zero', 'One']

ConfusionMatrixDisplay(confusion_matrix=cf_matrix, display_labels=categories).plot(cmap='Blues', colorbar=False)